# Milestone 2 - Model Inference

**Nama:** Rizqi Yudha Utama

**Batch:** 046

Notebook ini digunakan untuk mencoba **model final** (Random Forest hasil tuning) dari notebook `P1M2_rizqi_utama.ipynb` pada **data baru**, yaitu data yang tidak termasuk ke dalam data train maupun data test.

Data baru ditulis dalam **format asli** (belum di-scaling atau di-encoding), karena proses preprocessing sudah ada di dalam Pipeline model.

# 1. Import Libraries

In [14]:
import pandas as pd
import pickle

# 2. Load Model

In [15]:
# Load model final (Pipeline lengkap: preprocessing + Random Forest)
with open('model_final.pkl', 'rb') as file:
    model = pickle.load(file)

# 3. Loading Data Baru

Data baru dibuat dengan kolom yang sama seperti data asli sebelum preprocessing.

- Penulisan kategori dibuat **sama persis** dengan data training, contohnya merek ditulis huruf kecil (`volkswagen`) dan nama model ditulis lengkap dengan mereknya (`Volkswagen Golf`).
- Penjual biasanya tahu **tahun** mobilnya, jadi input-nya berupa tahun, lalu diubah menjadi **umur mobil** (`age`) dengan cara yang sama seperti saat cleaning, yaitu `2023 - year`.

In [16]:
data_baru = pd.DataFrame({
    'brand': ['volkswagen', 'volvo', 'opel', 'porsche'],
    'model': ['Volkswagen Golf', ' Volvo XC40', 'Opel Corsa', 'Porsche 911 GT3' ],
    'color': ['black', 'white', 'grey','silver'],
    'transmission_type': ['Manual', 'Automatic', 'Manual','Automatic'],
    'fuel_type': ['Petrol', 'Hybrid', 'Petrol','Petrol'],
    'year': [2018, 2022, 2012,2023],
    'mileage_in_km': [90000, 15000, 150000,20000],
    'power_ps': [150, 211, 90,510],
    'fuel_consumption_l_100km': [5.5, 2.0, 5.8,14.7]
})

# Hitung umur mobil, sama seperti saat cleaning
data_baru['age'] = 2023 - data_baru['year']

data_baru

,brand,model,color,transmission_type,fuel_type,year,mileage_in_km,power_ps,fuel_consumption_l_100km,age
0,volkswagen,Volkswagen Golf,black,Manual,Petrol,2018,90000,150,5.5,5
1,volvo,Volvo XC40,white,Automatic,Hybrid,2022,15000,211,2.0,1
2,opel,Opel Corsa,grey,Manual,Petrol,2012,150000,90,5.8,11
3,porsche,Porsche 911 GT3,silver,Automatic,Petrol,2023,20000,510,14.7,0


# 4. Prediksi Harga (Model Inferencing)

In [17]:
data_baru['Prediksi Harga (€)'] = model.predict(data_baru)

data_baru[['brand', 'model', 'year', 'mileage_in_km', 'power_ps', 'fuel_type', 'Prediksi Harga (€)']]

,brand,model,year,mileage_in_km,power_ps,fuel_type,Prediksi Harga (€)
0,volkswagen,Volkswagen Golf,2018,90000,150,Petrol,19473.343635
1,volvo,Volvo XC40,2022,15000,211,Hybrid,50165.467080
2,opel,Opel Corsa,2012,150000,90,Petrol,5200.243887
3,porsche,Porsche 911 GT3,2023,20000,510,Petrol,219264.146988


# 5. Kesimpulan Inference

Model berhasil memberikan estimasi harga untuk 4 mobil baru yang tidak ada di data train maupun data test:

- **Volkswagen Golf (2018, Petrol, 90.000 km, 150 PS)** diprediksi sekitar **€19.473**. Harga ini dekat dengan median harga Volkswagen umur 5 tahun di EDA (sekitar €21.000), jadi hasilnya sesuai dengan pola di data.
- **Volvo XC40 (2022, Hybrid, 15.000 km, 211 PS)** diprediksi sekitar **€50.165**. Masuk akal karena mobilnya masih sangat baru (umur 1 tahun), jarak tempuhnya rendah, dan memakai mesin hybrid. Di EDA, mobil yang masih baru dan mobil hybrid memang cenderung lebih mahal.
- **Opel Corsa (2012, Petrol, 150.000 km, 90 PS)** diprediksi sekitar **€5.200**, paling murah. Masuk akal karena mobilnya paling tua (umur 11 tahun), jarak tempuhnya paling jauh, tenaganya paling kecil, dan Opel adalah merek paling murah di antara 5 merek terbanyak di EDA.
- **Porsche 911 GT3 (2023, Petrol, 20.000 km, 510 PS)** diprediksi sekitar **€219.264**, paling mahal. Mobil ini sengaja ditambahkan untuk melihat bagaimana model menebak harga **mobil mewah**. Hasilnya cukup dekat dengan median harga mobil mewah dengan jarak tempuh 0-25 ribu km di EDA (sekitar **€190.000**).
  - Sebagai pembanding, iklan Porsche 911 GT3 tahun 2023 di Jerman saat ini (2026) sekitar **€180.000**. Prediksi model lebih tinggi, kemungkinan karena data training diambil tahun 2023 saat mobil ini masih baru, sedangkan iklan pembanding diambil saat mobilnya sudah berumur sekitar 3 tahun.
  - Nama model "911 GT3" kemungkinan tidak ada di data training, sehingga dianggap sebagai kategori "lainnya". Tapi model tetap bisa mengenali bahwa ini mobil mewah dari **merek, tenaga mesin, umur, dan jarak tempuhnya**.

**Kesimpulan:** hasil prediksi model **sesuai dengan logika dan insight dari EDA**: mobil yang lebih baru, jarak tempuhnya lebih rendah, dan tenaganya lebih besar diprediksi lebih mahal, dan merek mewah seperti Porsche diprediksi jauh lebih mahal dibanding merek biasa. Artinya model sudah bisa dipakai untuk memberikan **estimasi harga acuan awal** bagi penjual mobil bekas.

# 6. Perbandingan dengan Harga Iklan Nyata

Untuk mengecek apakah hasil prediksi masih masuk akal, prediksi model dibandingkan dengan harga iklan mobil serupa di situs jual-beli mobil bekas di Jerman.

**DISCLAIMER:** harga pembanding diambil dari iklan **tahun 2026**, sedangkan dataset diambil tahun **2023**. Jadi mobil di iklan pembanding sudah sekitar 3 tahun lebih dibanding saat dataset diambil. Wajar kalau harga iklan sekarang lebih rendah dari prediksi model. Perbandingan ini sifatnya **kasar**, hanya untuk melihat apakah prediksi masih di kisaran yang masuk akal.

**1. Volkswagen Golf (2018, bensin, 150 PS, 90.000 km)**
- Prediksi model: **€19.473**
- Iklan serupa: Golf 150 PS bensin dengan 106.000 km sekitar **€17.000**, dan Golf Sportsvan 150 PS dengan 62.899 km sekitar **€16.790**.
- Sumber: [12gebrauchtwagen.de - VW Golf 2018](https://www.12gebrauchtwagen.de/auto/vw/golf/y-2018)

**2. Volvo XC40 (2022, hybrid, 211 PS, 15.000 km)**
- Prediksi model: **€50.165**
- Iklan serupa: XC40 plug-in hybrid 2022 sekitar **€26.000 sampai €29.100**, tapi jarak tempuhnya jauh lebih tinggi (79.000 sampai 81.000 km). Rata-rata iklan XC40 2022 sekitar **€28.200**.
- Sumber: [autobild.de - Volvo XC40 2022](https://gebrauchtwagen.autobild.de/a-z/volvo/volvo-xc40/aus-2022/)

**3. Opel Corsa (2012, bensin, 90 PS, 150.000 km)**
- Prediksi model: **€5.200**
- Iklan serupa: harga Corsa 2012 berkisar **€1.700 sampai €6.990**. Corsa 87 PS dengan 131.000 km sekitar **€2.999**.
- Sumber: [autouncle.de - Opel Corsa 2012](https://www.autouncle.de/de/gebrauchtwagen/Opel/Corsa/y-2012)

**4. Porsche 911 GT3 (2023, bensin, 510 PS, 20.000 km)**
- Prediksi model: **€219.264**
- Iklan serupa: 911 GT3 2023 sekitar **€180.000** (dengan jarak tempuh 15.000 sampai 22.000 km).
- Sumber: [12gebrauchtwagen.de - Porsche 992 2023](https://www.12gebrauchtwagen.de/auto/porsche/992/y-2023)

**Kesimpulan perbandingan:**
- Prediksi **Golf** dan **Porsche** cukup dekat dengan harga iklan sekarang, sekitar 15% sampai 20% lebih tinggi.
- Prediksi **Opel Corsa** masih masuk di kisaran harga iklan sekarang, di bagian atasnya.
- Prediksi **Volvo XC40** paling jauh, sekitar 50% lebih tinggi. Kemungkinan karena iklan pembanding jarak tempuhnya jauh lebih tinggi, dan di data tahun 2023 mobil ini masih hampir baru.
- Semua prediksi **lebih tinggi** dari harga iklan sekarang. Ini sesuai dengan perbedaan waktu: model belajar dari harga tahun 2023, sedangkan harga mobil bekas terus turun seiring umur mobil.
- Akurasi model yang sebenarnya tetap dilihat dari hasil evaluasi di data test (**MAE sekitar €3.131**).

***Disclaimer: Sesuai batasan model, hasil ini adalah estimasi harga iklan berdasarkan data spesifikasi saja, belum memperhitungkan kondisi fisik mobil.***